In [5]:
import requests
import json
import pandas as pd
import time
from seleniumwire import webdriver  # 注意：是 seleniumwire 的 webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

In [3]:
def get_accept_enckey(timeout=30, headless=False):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )

    # seleniumwire_options 留默认即可；若想排除静态资源加速，可加 ignore_http_methods
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.header_overrides = {"Referer": 'https://webapi.cninfo.com.cn'}
    captured = {}
    try:
        driver.get("https://webapi.cninfo.com.cn/#/dataBrowse")

        deadline = time.time() + timeout
        while time.time() < deadline:
            for req in driver.requests:
                # 只挑接口类 XHR；截图中的 p_public0001 也包含 p_
                if req.response and "/p_" in req.url:
                    enckey = req.headers.get("Accept-Enckey")
                    if enckey:
                        captured = {"enckey": enckey, "url": req.url}
                        break
            if captured:
                break
            time.sleep(0.5)

        if not captured:
            raise TimeoutError(
                f"{timeout}s 内未捕获到 Accept-Enckey，"
                f"共记录请求 {len(driver.requests)} 个"
            )
        return captured
    finally:
        driver.quit()

In [23]:
headers = {'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/155.0.0.0 Safari/537.36'}
headers['Referer'] = 'https://webapi.cninfo.com.cn/'
headers['Host'] = 'webapi.cninfo.com.cn'

In [101]:
def ETF_PCF(headers,codes,start_date,end_date):   #### codes为六位数字的列表，可以只传入一个代码。date为形如yyyy-mm-dd的字符串
    result = get_accept_enckey()
    headers['accept-enckey'] = result['enckey']
    s1 = time.time()
    lists = []
    for c in codes:
        data = 'scode=' + c + '&sdate='+ start_date + '&edate=' + end_date
        url = 'https://webapi.cninfo.com.cn/api/fund/p_fund2605?'+ data
        jtext = json.loads(requests.post(url,headers=headers,data=data).text)['records']
        lists += jtext
        s2 = time.time()
        if s2 - s1 > 60*6:
            result = get_accept_enckey()
            headers['accept-enckey'] = result['enckey']
    return pd.DataFrame(lists).rename(columns={
    "SECCODE": "基金代码",
    "SECNAME": "基金简称",
    "F001D": "公告日期",
    "F002V": "股票代码",
    "F003V": "股票简称",
    "F004N": "股份数量",
    "F005V": "现金替代标志",
    "F006N": "现金替代溢价比例",
    "F007N": "固定替代金额"})

In [102]:
df = ETF_PCF(headers,codes=['158005','158001'],start_date='20261022',end_date='20261025')